# Analysis Pipeline — DeepLabCut vs. TopScan
**Memory Laboratory - Brain Institute (UFRN) | Technical Validation of Animal Tracking**

---

## Overview

This script performs the **quantitative and statistical analysis** of animal trajectories
captured by two tracking systems:

- **TopScan** — reference system (*gold standard*), coordinates already in millimetres.
- **DeepLabCut (DLC)** — ResNet-50 *pose estimation* model, pixel coordinates
  converted to cm during pre-processing.

The pipeline compares the two systems frame by frame, computes agreement metrics,
and generates visualisations and annotated videos for publication.

---

## Prerequisites

### Python dependencies
```bash
pip install pandas numpy scipy scikit-learn matplotlib seaborn pingouin similaritymeasures opencv-python
```

### Input files
This script consumes the output of **Preprocessing Pipeline — DeepLabCut vs. TopScan.ipynb**.
Make sure the processed `.txt` files are in the configured folders
before running.

### Expected folder structure on Google Drive
```
MyDrive/
└── conteudo/
    └── validacao_topscan/
        ├── TOPSCAN/
        │   └── TXT novo/          ← RATO_XX_topscan.txt files  (output of [1])
        ├── DEEPLABCUT/
        │   └── trajetoria_txt_dlc/ ← RATO_XX_dlc.txt files      (output of [1])
        ├── videos_ratinho/         ← original .mp4 videos
        └── resultados/
            └── videos/            ← annotated videos (auto-generated)
```

> **Note:** Adjust the paths in the `CONFIGURATION` section before running.

---

## Configuration Parameters

| Parameter | Default | Description |
|---|---|---|
| `FPS_VIDEO` | `30` | Video frame rate |
| `ARENA_LARGURA_CM` / `_ALTURA_CM` | `60.0` | Physical dimensions of the arena in cm |
| `IGNORAR_FRAMES_INICIAIS` | `150` | Frames discarded at the start (≈5 s of habituation) |
| `BODYPART_REFERENCIA` | `'centerbody'` | Body part used in statistics |
| `LIMITE_SALTO_DISTANCIA` | `100.0` | Maximum allowed jump between frames (pixels) |
| `PARAMS_SUAVIZACAO` | `window=11, order=3` | Savitzky-Golay filter parameters |
| `PARAMS_FILTRO_DBSCAN` | `eps=0.2, min_samples=10` | Clustering for outlier removal |
| `RATOS_PARA_GERAR_VIDEO` | `["RATO_15"]` | List of rats for video generation |

---

## Pipeline Flow

```
run_pipeline()
│
├── 1. load_data()
│       Detects TopScan and DLC files by name pattern (regex RATO_XX)
│
├── 2. process_trajectories()
│       For each rat and body part:
│       ├── IQR + DBSCAN → remove outliers
│       ├── Savitzky-Golay → kinematic smoothing
│       └── Conversion to cm (DLC pixels → cm via arena scale)
│
├── 3. plot_comparative_trajectories()
│       Overlaid trajectories colour-coded by time (TopScan vs DLC)
│
├── 4. plot_raw_trajectories()
│       Raw trajectories pre-processing for visual inspection
│
├── 5. define_rois() + verify_rois()
│       Defines circular regions of interest (ROI) for familiar/novel objects
│
├── 6. generate_videos()
│       Exports .mp4 videos with annotated trajectory frame by frame (via OpenCV)
│
├── 7. calculate_behavioral_exploration()
│       Classifies each frame as Familiar Object / Novel Object / Floor
│       Generates ethograms (broken bar) and computes Discrimination Index (DI)
│
└── 8. run_statistics()
        ├── 4.1  Metrics table: RMSE, MAE, Fréchet, DTW, Pearson
        ├── 4.2  Spatial heatmap of positional error
        ├── 4.3  ANOVA by behavioural state + Bonferroni post-hoc
        ├── 4.4  DI: Shapiro-Wilk + paired t-test (TopScan vs DLC)
        ├── 4.5  Bland-Altman (DI agreement)
        ├── 4.6  Temporal cross-correlation of displacements
        ├── 4.7  Error by zone (centre vs. periphery)
        ├── 4.8  Velocity KDE + K-S test (TopScan vs DLC distribution)
        └── 4.9  4-panel dashboard per animal (trajectory, error, heatmap, velocity)
```

---

## How to Run in Google Colab

1. Mount Google Drive:
   ```python
   from google.colab import drive
   drive.mount('/content/drive')
   ```
2. Install dependencies (setup cell — run once).
3. Adjust paths in the `CONFIGURATION` section.
4. Call `run_pipeline()` at the end of the notebook.

To run a single stage in isolation, call the corresponding function directly
after executing the imports, configuration, and processing cells.

---

#CONNECTION TO YOUR DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

##IMPORTS

In [ ]:
import os
import re

import cv2
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pingouin as pg
import seaborn as sns
import similaritymeasures
from scipy.signal import correlate, savgol_filter
from scipy.signal.windows import hann
from scipy.stats import ks_2samp, pearsonr, shapiro, ttest_rel
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

##CONFIGURATION

In [ ]:
DIRETORIO_TOPSCAN_TXT    = '/conteudo/validacao_topscan/TOPSCAN/TXT novo/'
DIRETORIO_DLC_TXT        = '/conteudo/validacao_topscan/DEEPLABCUT/trajetoria_txt_dlc/'
DIRETORIO_VIDEOS         = '/conteudo/validacao_topscan/videos_ratinho/'
DIRETORIO_OUTPUT_VIDEOS  = '/conteudo/validacao_topscan/resultados/videos/'

FPS_VIDEO        = 30
ARENA_LARGURA_CM = 60.0
ARENA_ALTURA_CM  = 60.0

# Pixel dimensions (width, height) for each video
DIMENSOES_VIDEOS = {
    "RATO_02": (310, 240), "RATO_05": (310, 240), "RATO_07": (330, 240),
    "RATO_09": (330, 240), "RATO_10": (330, 240), "RATO_12": (330, 224),
    "RATO_15": (330, 224), "RATO_16": (330, 240), "RATO_19": (330, 240),
    "RATO_42": (330, 224),
}

# Filtering and smoothing parameters
PARAMS_SUAVIZACAO      = {'window_length': 11, 'polyorder': 3}
PARAMS_FILTRO_DBSCAN   = {'eps': 0.2, 'min_samples': 10}
LIMITE_SALTO_DISTANCIA = 100.0

# Behaviour
NOMES_OBJETOS_FAMILIARES = ['OBJF', 'OBJA']
NOMES_OBJETOS_NOVOS      = ['OBJJ', 'OBJC', 'OBJN1']
TOPSCAN_AREA_MAP_VIDEO   = {'OBJF': 'Familiar Object', 'OBJJ': 'Novel Object', 'Floor': 'Floor'}

# Analysis
IGNORAR_FRAMES_INICIAIS = 150
BODYPART_REFERENCIA     = 'centerbody'
BODYPART_LEGENDA        = 'nose'
BODYPARTS_ANALISE       = ['centerbody', 'nose']

# Rats for video generation (empty list = all)
RATOS_PARA_GERAR_VIDEO   = ["RATO_15"]
SISTEMAS_PARA_GERAR_VIDEO = ["topscan", "dlc"]

# ROI templates (configure for the experiment)
ROI_TEMPLATES = {
    "config_A": {
        "Familiar": {'tipo': 'circulo', 'x': 92,  'y': 115, 'raio': 17},
        "Novo":     {'tipo': 'circulo', 'x': 185, 'y': 118, 'raio': 17},
    },
    "config_B": {
        "Familiar": {'tipo': 'circulo', 'x': 105, 'y': 128, 'raio': 17},
        "Novo":     {'tipo': 'circulo', 'x': 190, 'y': 125, 'raio': 17},
    },
    "config_C": {
        "Familiar": {'tipo': 'circulo', 'x': 98,  'y': 125, 'raio': 17},
        "Novo":     {'tipo': 'circulo', 'x': 188, 'y': 120, 'raio': 17},
    },
}

RATO_ROI_CONFIG = {
    "RATO_02": "config_C", "RATO_05": "config_C", "RATO_07": "config_B",
    "RATO_09": "config_A", "RATO_10": "config_B", "RATO_12": "config_A",
    "RATO_15": "config_A", "RATO_16": "config_B", "RATO_19": "config_B",
    "RATO_42": "config_A",
}

norm_tempo_color = mcolors.Normalize(vmin=0, vmax=300)

##GLOBAL VISUALISATION STYLE

In [ ]:
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        13,
    'axes.titlesize':   19,
    'axes.titleweight': 'bold',
    'axes.labelsize':   15,
    'xtick.labelsize':  13,
    'ytick.labelsize':  13,
    'legend.fontsize':  12,
    'figure.dpi':       100,
    'savefig.dpi':      150,
    'axes.grid':        True,
    'grid.linestyle':   ':',
    'grid.alpha':       0.5,
})

##FUNCTIONS

###IO

In [ ]:
def load_dlc_txt(path):
    if not os.path.exists(path):
        print(f"Error: DLC TXT not found at '{path}'")
        return None
    try:
        df = pd.read_csv(path, sep=' ', header=0)
        if not all(c in df.columns for c in ['Frame', 'X', 'Y']):
            print(f"Error: expected columns ['Frame','X','Y'] missing in {path}")
            return None
        return df
    except Exception as e:
        print(f"Error loading DLC TXT '{path}': {e}")
        return None


def load_topscan_txt(path):
    expected = ['FrameNum', 'CenterX(mm)', 'CenterY(mm)', 'Areas']
    if not os.path.exists(path):
        print(f"Error: TopScan TXT not found at '{path}'")
        return None
    try:
        df = pd.read_csv(path, sep=r'\s+', header=0, skipinitialspace=True)
        col_map = {
            'CenterX': 'CenterX(mm)', 'CenterY': 'CenterY(mm)',
            'Area': 'Areas', 'Frame': 'FrameNum',
        }
        df.rename(columns={k: v for k, v in col_map.items() if k in df.columns and v not in df.columns}, inplace=True)
        if not all(c in df.columns for c in expected):
            print(f"Error: expected columns {expected} missing in {path}. Found: {df.columns.tolist()}")
            return None
        return df
    except Exception as e:
        print(f"Error loading TopScan TXT '{path}': {e}")
        return None


def extract_rat_id(text):
    match = re.search(r'R[ _\-]?A?T?O?[ _\-]?(\d+)', text.upper())
    return f"RATO_{match.group(1).zfill(2)}" if match else None

###PROCESSING

In [ ]:
def add_time_column(df, col_frame, fps):
    df = df.copy()
    df['time'] = pd.to_numeric(df[col_frame], errors='coerce') / float(fps) if fps > 0 else np.nan
    return df


def estimate_filter_params(x, y):
    Q1_x, Q3_x = np.percentile(x, [25, 75])
    Q1_y, Q3_y = np.percentile(y, [25, 75])
    spread = np.mean([Q3_x - Q1_x, Q3_y - Q1_y])

    if   spread > 150: iqr_limit = 0.4
    elif spread > 120: iqr_limit = 0.6
    elif spread > 90:  iqr_limit = 0.8
    elif spread > 60:  iqr_limit = 1.0
    elif spread > 30:  iqr_limit = 1.2
    else:              iqr_limit = 1.5

    distances = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
    distances = distances[~np.isnan(distances)]
    jump_p95 = np.percentile(distances, 95)
    jump_p99 = np.percentile(distances, 99)
    jump_limit = max(jump_p95 * 1.5, jump_p99, np.mean(distances) * 3)

    std_dist   = np.nanstd(distances)
    mean_dist  = np.nanmean(distances)
    eps        = min(0.25, max(0.05, mean_dist + std_dist * 0.5))
    min_samples = 5 if len(x) < 200 else (10 if len(x) < 1000 else 15)

    return {
        "limite_iqr": round(iqr_limit, 2),
        "params_dbscan": {"eps": round(eps, 3), "min_samples": min_samples},
        "limite_salto": round(jump_limit, 2),
    }


def _filter_trajectory(data, col_x, col_y, limite_iqr, params_dbscan, limite_salto, suffix):
    data[col_x] = pd.to_numeric(data[col_x], errors='coerce')
    data[col_y] = pd.to_numeric(data[col_y], errors='coerce')
    data.dropna(subset=[col_x, col_y], inplace=True)
    if data.empty:
        return None

    Q1_x, Q3_x = data[col_x].quantile([0.25, 0.75])
    Q1_y, Q3_y = data[col_y].quantile([0.25, 0.75])
    data = data[
        data[col_x].between(Q1_x - limite_iqr * (Q3_x - Q1_x), Q3_x + limite_iqr * (Q3_x - Q1_x)) &
        data[col_y].between(Q1_y - limite_iqr * (Q3_y - Q1_y), Q3_y + limite_iqr * (Q3_y - Q1_y))
    ]
    if data.empty:
        return None

    coords_scaled = StandardScaler().fit_transform(data[[col_x, col_y]].values)
    labels = DBSCAN(**params_dbscan).fit_predict(coords_scaled)
    data = data[labels != -1].copy()
    if data.empty:
        return None

    distance = np.sqrt(data[col_x].diff()**2 + data[col_y].diff()**2).fillna(0)
    data = data[distance <= limite_salto]

    return data.rename(columns={col_x: f'x_raw_filtrado{suffix}', col_y: f'y_raw_filtrado{suffix}'})


def process_topscan_filtered(path, limite_iqr, params_dbscan, limite_salto):
    data = load_topscan_txt(path)
    if data is None or data.empty:
        return None
    return _filter_trajectory(data, 'CenterX(mm)', 'CenterY(mm)', limite_iqr, params_dbscan, limite_salto, '')


def process_dlc_filtered(path, limite_iqr, params_dbscan, limite_salto):
    data = load_dlc_txt(path)
    if data is None or data.empty:
        return None
    return _filter_trajectory(data, 'X', 'Y', limite_iqr, params_dbscan, limite_salto, '')


def smooth_with_savgol(df, col_x='x', col_y='y', window_length=11, polyorder=3):
    df = df.copy()
    for col in [col_x, col_y]:
        if col not in df.columns:
            continue
        valid_data = df[col].dropna()
        w = int(window_length)
        if w % 2 == 0:
            w -= 1
        p = min(int(polyorder), w - 1)
        if len(valid_data) >= w and len(valid_data) > p:
            df.loc[valid_data.index, col] = savgol_filter(valid_data.values, w, p)
    return df


def determine_dlc_exploration(row, definicoes, col_x, col_y):
    if pd.isna(row[col_x]) or pd.isna(row[col_y]):
        return 'chao'
    point = np.array([row[col_x], row[col_y]])
    for name, info in definicoes.items():
        if info.get('tipo') == 'circulo':
            if np.linalg.norm(point - np.array([info['x'], info['y']])) <= info['raio']:
                return f"objeto_{name}"
    return 'chao'


def _prepare_dlc_data_for_video(path, fps, object_definitions=None):
    data = load_dlc_txt(path)
    if data is None or data.empty:
        return None
    data['X'] = pd.to_numeric(data['X'], errors='coerce')
    data['Y'] = pd.to_numeric(data['Y'], errors='coerce')
    df = data.rename(columns={'Frame': 'FrameNum', 'X': 'x_video_px', 'Y': 'y_video_px'})
    df = add_time_column(df, 'FrameNum', fps)
    if object_definitions:
        df['exploration_type'] = df.apply(
            determine_dlc_exploration, axis=1,
            definicoes=object_definitions, col_x='x_video_px', col_y='y_video_px',
        )
    else:
        df['exploration_type'] = 'N/A'
    return df


def _sync_data_cm(rat_id, results_cm, bodypart=BODYPART_REFERENCIA, ignore_frames=IGNORAR_FRAMES_INICIAIS):
    """Retrieves, synchronises, and filters initial frames. Returns df_analysis or None."""
    try:
        df_ts  = results_cm[rat_id][bodypart]['topscan'].copy()
        df_dlc = results_cm[rat_id][bodypart]['dlc'].copy()
    except KeyError:
        return None
    df = pd.merge(df_ts, df_dlc, on='FrameNum', suffixes=('_topscan', '_dlc')).dropna()
    df = df[df['FrameNum'] >= ignore_frames].copy()
    return df if not df.empty else None

###VISUALISATION

In [ ]:
def plot_temporal_trajectory_scatter(ax, x_coords, y_coords, time_coords,
                                     cmap_name='viridis', norm_colors=None,
                                     titulo="", label_x="X", label_y="Y",
                                     base_line_color='gray', base_line_alpha=0.5,
                                     scatter_size=10, scatter_alpha=0.7):
    x = np.asarray(x_coords)
    y = np.asarray(y_coords)
    t = np.asarray(time_coords)
    if len(x) == 0:
        ax.set_title(titulo); ax.set_xlabel(label_x); ax.set_ylabel(label_y)
        return None
    valid_t = t[~np.isnan(t)]
    if norm_colors is None and len(valid_t) > 1:
        norm_colors = mcolors.Normalize(vmin=np.min(valid_t), vmax=np.max(valid_t))
    ax.plot(x, y, color=base_line_color, alpha=base_line_alpha)
    ax.scatter(x, y, c=t, cmap=cmap_name, norm=norm_colors, s=scatter_size, alpha=scatter_alpha, zorder=3)
    ax.set_title(titulo); ax.set_xlabel(label_x); ax.set_ylabel(label_y)
    if norm_colors:
        sm = cm.ScalarMappable(cmap=plt.get_cmap(cmap_name), norm=norm_colors)
        sm.set_array([])
        return sm
    return None


def _plot_broken_bar(periods, names, title="Exploration Periods", figure_size=(10, 4)):
    if len(periods) != len(names):
        print("Error: number of period lists differs from number of names.")
        return
    fig, ax = plt.subplots(figsize=figure_size)
    colors = plt.colormaps['viridis'].resampled(len(names))
    for i, p in enumerate(periods):
        if p:
            ax.broken_barh(p, (i - 0.4, 0.8), facecolors=colors(i), label=names[i])
    ax.set_yticks(range(len(names)))
    ax.set_yticklabels(names)
    ax.set_xlabel("Video Frames")
    ax.set_ylabel("Objects / Zones")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    plt.show()


def _style_trajectory_axis(ax, title, xlim, ylim, fontsize_title=19, fontsize_label=15, fontsize_tick=13):
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_aspect('equal', adjustable='box')
    ax.tick_params(axis='both', which='major', labelsize=fontsize_tick)
    ax.set_xlabel("X (cm)", fontsize=fontsize_label)
    ax.set_ylabel("Y (cm)", fontsize=fontsize_label)
    ax.set_title(title, fontsize=fontsize_title, fontweight='bold')

###VÍDEO

In [ ]:
def _annotate_video_with_trajectory_and_legend(
    input_path, df_coords, col_x, col_y, col_frame_sync, output_path,
    rois_to_draw=None, legend_text_fn=None, output_fps=None,
    trajectory_color=(0, 255, 0), point_color=(0, 0, 255), legend_color=(255, 255, 255),
    font_scale=0.4, thickness=1, legend_position=(10, 30), n_trail_points=100,
):
    print(f"Annotating: '{output_path}'")
    if not os.path.exists(input_path):
        print(f"ERROR: video not found at '{input_path}'")
        return False

    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print(f"ERROR: could not open '{input_path}'")
        return False

    fps_orig  = cap.get(cv2.CAP_PROP_FPS)
    output_fps = output_fps or (fps_orig if fps_orig > 1 else 30.0)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    if width == 0 or height == 0:
        print("ERROR: invalid video dimensions.")
        cap.release()
        return False

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), output_fps, (width, height))
    if not out.isOpened():
        print("ERROR: could not create output video.")
        cap.release()
        return False

    roi_colors = {'Familiar': (255, 100, 100), 'Novo': (100, 100, 255)}
    current_frame = 0
    trail = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if rois_to_draw:
            for name, info in rois_to_draw.items():
                if info.get('tipo') == 'circulo':
                    color = roi_colors.get(name, (0, 255, 255))
                    cv2.circle(frame, (info['x'], info['y']), info['raio'], color, 2)
                    cv2.putText(frame, name, (info['x'] - info['raio'], info['y'] - info['raio'] - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

        row = df_coords[df_coords[col_frame_sync] == current_frame]
        if not row.empty:
            xv = row[col_x].iloc[0]
            yv = row[col_y].iloc[0]
            if not (pd.isna(xv) or pd.isna(yv)):
                pt = (int(xv), int(yv))
                trail.append(pt)
                cv2.circle(frame, pt, 5, point_color, -1)

        points = trail[-n_trail_points:] if isinstance(n_trail_points, int) else trail
        for i in range(1, len(points)):
            if points[i - 1] and points[i]:
                cv2.line(frame, points[i - 1], points[i], trajectory_color, 2)

        if legend_text_fn:
            frame_data = row.iloc[0] if not row.empty else None
            text = legend_text_fn(frame_data, current_frame)
            if text:
                cv2.putText(frame, text, legend_position, cv2.FONT_HERSHEY_SIMPLEX,
                            font_scale, legend_color, thickness, cv2.LINE_AA)

        out.write(frame)
        current_frame += 1

    cap.release()
    out.release()
    print(f"  Video saved at: {output_path}")
    return True

##STAGE 0: LOAD DATA (Detect available files)

In [ ]:
def load_data():
    dlc_files     = {bp: {} for bp in BODYPARTS_ANALISE}
    topscan_files = {}

    for fname in os.listdir(DIRETORIO_DLC_TXT):
        for bp in BODYPARTS_ANALISE:
            if bp in fname.lower():
                rid = extract_rat_id(fname)
                if rid:
                    dlc_files[bp][rid] = fname

    for fname in os.listdir(DIRETORIO_TOPSCAN_TXT):
        if fname.lower().endswith('.txt'):
            rid = extract_rat_id(fname)
            if rid:
                num = rid.split("_")[1]
                topscan_files[rid] = {
                    "arquivo_txt": fname,
                    "video_cortado_nome": f"experimento{int(num)}.mp4",
                }

    rats_dlc     = set(dlc_files[BODYPART_REFERENCIA].keys()) | set(dlc_files.get(BODYPART_LEGENDA, {}).keys())
    rats_topscan = set(topscan_files.keys())
    common_rats  = rats_dlc & rats_topscan

    topscan_files = {r: topscan_files[r] for r in common_rats}
    dlc_files     = {bp: {r: f for r, f in files.items() if r in common_rats}
                     for bp, files in dlc_files.items()}

    print(f"Valid rats ({len(common_rats)}): {sorted(common_rats)}")
    return common_rats, topscan_files, dlc_files

##STAGE 1: PROCESS TRAJECTORIES

###STAGE 1.1: CONVERSION TO CMs

In [ ]:
def process_trajectories(common_rats, topscan_files, dlc_files):
    """
    Filters, spatially aligns, and converts to centimetres.
    Returns results_cm[rat_id][bodypart] = {'topscan': df, 'dlc': df}
    """
    results = {}

    for rat_id in sorted(common_rats):
        print(f"\nProcessing {rat_id}...")
        results[rat_id] = {}

        for bodypart in BODYPARTS_ANALISE:
            if rat_id not in dlc_files.get(bodypart, {}):
                continue

            path_ts  = os.path.join(DIRETORIO_TOPSCAN_TXT, topscan_files[rat_id]["arquivo_txt"])
            path_dlc = os.path.join(DIRETORIO_DLC_TXT, dlc_files[bodypart][rat_id])

            raw_ts_data  = load_topscan_txt(path_ts)
            raw_dlc_data = load_dlc_txt(path_dlc)
            if raw_ts_data is None or raw_dlc_data is None:
                continue

            filter_ts  = estimate_filter_params(raw_ts_data['CenterX(mm)'].dropna().values,
                                                raw_ts_data['CenterY(mm)'].dropna().values)
            filter_dlc = estimate_filter_params(raw_dlc_data['X'].dropna().values,
                                                raw_dlc_data['Y'].dropna().values)

            df_ts_filt  = process_topscan_filtered(path_ts,  **filter_ts)
            df_dlc_filt = process_dlc_filtered(path_dlc, **filter_dlc)
            if df_ts_filt is None or df_dlc_filt is None:
                continue

            df_dlc_filt.rename(columns={'Frame': 'FrameNum'}, inplace=True, errors='ignore')

            # Spatial alignment (TopScan mm -> DLC pixels)
            df_sync = pd.merge(df_ts_filt, df_dlc_filt, on='FrameNum', suffixes=('_ts', '_dlc')).dropna()
            if df_sync.empty:
                continue

            m_x, c_x = np.polyfit(df_sync['x_raw_filtrado_ts'], df_sync['x_raw_filtrado_dlc'], 1)
            m_y, c_y = np.polyfit(df_sync['y_raw_filtrado_ts'], df_sync['y_raw_filtrado_dlc'], 1)

            df_ts_filt['x_aligned_px'] = df_ts_filt['x_raw_filtrado'] * m_x + c_x
            df_ts_filt['y_aligned_px'] = df_ts_filt['y_raw_filtrado'] * m_y + c_y

            # Unified scale (cm/pixel)
            all_x = pd.concat([df_dlc_filt['x_raw_filtrado'], df_ts_filt['x_aligned_px']]).dropna()
            all_y = pd.concat([df_dlc_filt['y_raw_filtrado'], df_ts_filt['y_aligned_px']]).dropna()
            range_px = max(all_x.max() - all_x.min(), all_y.max() - all_y.min())
            scale    = ARENA_LARGURA_CM / float(range_px) if range_px > 0 else 0
            gmin_x, gmin_y = all_x.min(), all_y.min()

            # Convert to cm
            df_dlc_cm = df_dlc_filt.copy()
            df_dlc_cm['x_cm'] = (df_dlc_cm['x_raw_filtrado'] - gmin_x) * scale
            df_dlc_cm['y_cm'] = (df_dlc_cm['y_raw_filtrado'] - gmin_y) * scale
            df_dlc_cm['x_px'] = df_dlc_cm['x_raw_filtrado']
            df_dlc_cm['y_px'] = df_dlc_cm['y_raw_filtrado']

            df_ts_cm = df_ts_filt.copy()
            df_ts_cm['x_cm'] = (df_ts_cm['x_aligned_px'] - gmin_x) * scale
            df_ts_cm['y_cm'] = (df_ts_cm['y_aligned_px'] - gmin_y) * scale
            df_ts_cm['x_px'] = df_ts_cm['x_aligned_px']
            df_ts_cm['y_px'] = df_ts_cm['y_aligned_px']

            df_dlc_final = smooth_with_savgol(df_dlc_cm, 'x_cm', 'y_cm', **PARAMS_SUAVIZACAO)
            df_ts_final  = smooth_with_savgol(df_ts_cm,  'x_cm', 'y_cm', **PARAMS_SUAVIZACAO)
            df_dlc_final = add_time_column(df_dlc_final, 'FrameNum', FPS_VIDEO)
            df_ts_final  = add_time_column(df_ts_final,  'FrameNum', FPS_VIDEO)

            results[rat_id][bodypart] = {'topscan': df_ts_final, 'dlc': df_dlc_final}
            print(f"  {rat_id} ({bodypart}): scale={scale:.4f} cm/px")

    print("\nProcessing complete.")
    return results

###STAGE 1.2: COMPARATIVE TRAJECTORY PLOTS (Filtered, in cm)

In [ ]:
def plot_comparative_trajectories(results_cm):
    print("\n--- Comparative Trajectories (cm) ---")
    for rat_id, bp_data in results_cm.items():
        if BODYPART_REFERENCIA not in bp_data:
            continue
        data   = bp_data[BODYPART_REFERENCIA]
        df_ts  = data.get('topscan')
        df_dlc = data.get('dlc')

        fig, axes = plt.subplots(1, 2, figsize=(16, 8), constrained_layout=True)
        fig.suptitle(f"Trajectory Comparison — {rat_id}", fontsize=22)

        for ax, df, cmap, title in [
            (axes[0], df_ts,  'Reds',    "TopScan"),
            (axes[1], df_dlc, 'Purples', "DeepLabCut"),
        ]:
            if df is not None and not df.empty:
                sm = plot_temporal_trajectory_scatter(
                    ax, df['x_cm'], df['y_cm'], df.get('time', np.zeros(len(df))),
                    cmap_name=cmap, norm_colors=norm_tempo_color,
                    label_x="X (cm)", label_y="Y (cm)",
                )
                if sm:
                    cbar = fig.colorbar(sm, ax=ax, shrink=0.8)
                    cbar.set_label("Time (s)", fontsize=14)
                    cbar.ax.tick_params(labelsize=12)
            _style_trajectory_axis(ax, title, (0, ARENA_LARGURA_CM), (0, ARENA_ALTURA_CM))

        plt.show()

###STAGE 1.3: RAW TRAJECTORY PLOTS

In [ ]:
def plot_raw_trajectories(common_rats, topscan_files, dlc_files):
    print("\n--- Raw Trajectories (unfiltered) ---")
    for rat_id in sorted(common_rats):
        path_ts  = os.path.join(DIRETORIO_TOPSCAN_TXT, topscan_files[rat_id]["arquivo_txt"])
        path_dlc = os.path.join(DIRETORIO_DLC_TXT, dlc_files[BODYPART_REFERENCIA].get(rat_id, ''))

        df_ts  = load_topscan_txt(path_ts)
        df_dlc = load_dlc_txt(path_dlc)
        if df_ts is None or df_dlc is None:
            continue

        col_frame_ts = next((c for c in ['FrameNum', 'Frame'] if c in df_ts.columns), df_ts.columns[0])
        df_ts  = add_time_column(df_ts,  col_frame_ts, FPS_VIDEO)
        df_dlc = add_time_column(df_dlc, 'Frame',      FPS_VIDEO)

        fig, axes = plt.subplots(1, 2, figsize=(16, 7), layout='constrained')
        fig.suptitle(f"Raw Trajectories — {rat_id}", fontsize=22)

        df_plot_ts = df_ts.dropna(subset=['CenterX(mm)', 'CenterY(mm)'])
        if not df_plot_ts.empty:
            plot_temporal_trajectory_scatter(axes[0],
                df_plot_ts['CenterX(mm)'], df_plot_ts['CenterY(mm)'], df_plot_ts['time'],
                cmap_name='Reds', label_x="X (mm)", label_y="Y (mm)")
            xmin, xmax = df_plot_ts['CenterX(mm)'].min(), df_plot_ts['CenterX(mm)'].max()
            ymin, ymax = df_plot_ts['CenterY(mm)'].min(), df_plot_ts['CenterY(mm)'].max()
            axes[0].set_xlim(xmin - 5, xmax + 5)
            axes[0].set_ylim(ymin - 5, ymax + 5)
            axes[0].set_box_aspect(480 / 720)

        axes[0].set_title("TopScan", fontsize=19, fontweight='bold')
        axes[0].set_xlabel("X (mm)", fontsize=15)
        axes[0].set_ylabel("Y (mm)", fontsize=15)

        df_plot_dlc = df_dlc.dropna(subset=['X', 'Y'])
        if not df_plot_dlc.empty:
            plot_temporal_trajectory_scatter(axes[1],
                df_plot_dlc['X'], df_plot_dlc['Y'], df_plot_dlc['time'],
                cmap_name='Purples', label_x="X (pixels)", label_y="Y (pixels)")
            xmin, xmax = df_plot_dlc['X'].min(), df_plot_dlc['X'].max()
            ymin, ymax = df_plot_dlc['Y'].min(), df_plot_dlc['Y'].max()
            axes[1].set_xlim(xmin - 5, xmax + 5)
            axes[1].set_ylim(ymin - 5, ymax + 5)
            if rat_id in DIMENSOES_VIDEOS:
                w, h = DIMENSOES_VIDEOS[rat_id]
                axes[1].set_box_aspect(h / w)

        axes[1].set_title("DeepLabCut", fontsize=19, fontweight='bold')
        axes[1].set_xlabel("X (pixels)", fontsize=15)
        axes[1].set_ylabel("Y (pixels)", fontsize=15)

        plt.show()

##STAGE 2: VIDEOS (ROIs, preparation and generation)

In [ ]:
def define_rois(common_rats):
    rois = {}
    for rat_id, config_name in RATO_ROI_CONFIG.items():
        if rat_id not in common_rats:
            continue
        if config_name in ROI_TEMPLATES:
            rois[rat_id] = ROI_TEMPLATES[config_name]
            print(f"  {rat_id} -> {config_name}")
        else:
            print(f"ERROR: configuration '{config_name}' does not exist in ROI_TEMPLATES.")
    return rois


def verify_rois(defined_rois, topscan_files):
    print("\n--- Visual verification of ROIs ---")
    import random
    for rat_id, rois in defined_rois.items():
        info = topscan_files.get(rat_id)
        if not info:
            continue
        video_path = os.path.join(DIRETORIO_VIDEOS, info["video_cortado_nome"])
        if not os.path.exists(video_path):
            print(f"  ERROR: video not found at '{video_path}'")
            continue

        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            continue
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.set(cv2.CAP_PROP_POS_FRAMES, random.randint(int(total * 0.1), int(total * 0.9)))
        ret, frame = cap.read()
        cap.release()
        if not ret:
            continue

        for name, roi_info in rois.items():
            if roi_info.get('tipo') == 'circulo':
                color = (255, 100, 100) if "Familiar" in name else (100, 100, 255)
                cv2.circle(frame, (roi_info['x'], roi_info['y']), roi_info['raio'], color, 2)
                cv2.putText(frame, name, (roi_info['x'] + roi_info['raio'] + 5, roi_info['y'] + 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        plt.figure(figsize=(10, 8))
        plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        plt.title(f"ROI — {rat_id}")
        plt.axis('off')
        plt.show()


def _legend_exploration(frame_data, frame_num):
    if frame_data is not None and 'exploration_type' in frame_data and pd.notna(frame_data['exploration_type']):
        return f"Frame: {frame_num} | {frame_data['exploration_type'].replace('objeto_', '')}"
    return f"Frame: {frame_num}"


def generate_videos(common_rats, topscan_files, dlc_files, results_cm, defined_rois):
    os.makedirs(DIRETORIO_OUTPUT_VIDEOS, exist_ok=True)
    rats = RATOS_PARA_GERAR_VIDEO if RATOS_PARA_GERAR_VIDEO else sorted(common_rats)

    for rat_id in rats:
        if rat_id not in common_rats:
            print(f"Warning: '{rat_id}' is not in the common rats list.")
            continue

        info_ts = topscan_files.get(rat_id)
        rois    = defined_rois.get(rat_id)
        if not info_ts or not rois:
            print(f"  ERROR: missing data for '{rat_id}'.")
            continue

        video_path = os.path.join(DIRETORIO_VIDEOS, info_ts["video_cortado_nome"])
        if not os.path.exists(video_path):
            print(f"  ERROR: video not found at '{video_path}'.")
            continue

        try:
            df_ts_proc  = results_cm[rat_id][BODYPART_REFERENCIA]['topscan'].copy()
            df_dlc_proc = results_cm[rat_id][BODYPART_REFERENCIA]['dlc'].copy()
        except KeyError:
            print(f"  Processed data missing for {rat_id}. Skipping.")
            continue

        df_ts_video  = df_ts_proc.rename(columns={"x_px": "x_video_px", "y_px": "y_video_px"})
        df_dlc_video = df_dlc_proc.rename(columns={"x_px": "x_video_px", "y_px": "y_video_px"})

        if 'topscan' in SISTEMAS_PARA_GERAR_VIDEO:
            df_raw_ts = load_topscan_txt(os.path.join(DIRETORIO_TOPSCAN_TXT, info_ts["arquivo_txt"]))
            if df_raw_ts is not None and 'Areas' in df_raw_ts.columns:
                df_raw_ts['exploration_type'] = df_raw_ts['Areas'].map(TOPSCAN_AREA_MAP_VIDEO).fillna('Floor')
                df_ts_video = pd.merge(df_ts_video[['FrameNum', 'x_video_px', 'y_video_px', 'time']],
                                       df_raw_ts[['FrameNum', 'exploration_type']], on='FrameNum', how='left')
            output_name = os.path.join(DIRETORIO_OUTPUT_VIDEOS,
                                       f"{os.path.splitext(info_ts['video_cortado_nome'])[0]}_TopScan.mp4")
            _annotate_video_with_trajectory_and_legend(
                video_path, df_ts_video, 'x_video_px', 'y_video_px', 'FrameNum', output_name,
                rois_to_draw=rois, legend_text_fn=_legend_exploration,
            )

        if 'dlc' in SISTEMAS_PARA_GERAR_VIDEO:
            info_dlc = dlc_files.get(BODYPART_LEGENDA, {}).get(rat_id)
            if info_dlc:
                df_status = _prepare_dlc_data_for_video(
                    os.path.join(DIRETORIO_DLC_TXT, info_dlc), FPS_VIDEO, object_definitions=rois)
                if df_status is not None:
                    df_dlc_video = pd.merge(df_dlc_video[['FrameNum', 'x_video_px', 'y_video_px', 'time']],
                                            df_status[['FrameNum', 'exploration_type']], on='FrameNum', how='left')
            output_name = os.path.join(DIRETORIO_OUTPUT_VIDEOS,
                                       f"{os.path.splitext(info_ts['video_cortado_nome'])[0]}_DLC.mp4")
            _annotate_video_with_trajectory_and_legend(
                video_path, df_dlc_video, 'x_video_px', 'y_video_px', 'FrameNum', output_name,
                rois_to_draw=rois, legend_text_fn=_legend_exploration,
            )

##STAGE 3: BEHAVIOURAL EXPLORATION (Broken Bar)

In [ ]:
def _generate_periods(df_exp):
    if df_exp.empty:
        return []
    periods = []
    start, last = None, None
    df_exp = df_exp.sort_values('time').dropna(subset=['time'])
    for _, row in df_exp.iterrows():
        t, kind = row['time'], row['exploration_type']
        if kind != last:
            if last is not None and start is not None:
                dur = max(0, t - start - 1 / FPS_VIDEO)
                if dur > 1e-9:
                    periods.append((start, dur, last))
            start, last = t, kind
    if last is not None and start is not None and not df_exp.empty:
        dur = max(0, df_exp['time'].max() - start - 1 / FPS_VIDEO)
        if dur > 1e-9:
            periods.append((start, dur, last))
    return periods


def calculate_behavioral_exploration(common_rats, topscan_files, dlc_files, defined_rois):
    saved_times = {}
    colors = {'objeto_Familiar': '#cc5500', 'objeto_Novo': '#5472cc', 'chao': '#E0E0E0'}

    for rat_id in common_rats:
        info_ts  = topscan_files.get(rat_id)
        info_dlc = dlc_files.get(BODYPART_LEGENDA, {}).get(rat_id)
        rois     = defined_rois.get(rat_id)
        if not all([info_ts, info_dlc, rois]):
            continue

        # --- TopScan ---
        df_ts = load_topscan_txt(os.path.join(DIRETORIO_TOPSCAN_TXT, info_ts['arquivo_txt']))
        if df_ts is None or 'Areas' not in df_ts.columns:
            continue
        df_ts = add_time_column(df_ts, 'FrameNum', FPS_VIDEO)
        df_ts['exploration_type'] = df_ts['Areas'].apply(
            lambda a: 'objeto_Familiar' if a in NOMES_OBJETOS_FAMILIARES
                      else ('objeto_Novo' if a in NOMES_OBJETOS_NOVOS else 'chao')
        )

        # --- DLC ---
        df_dlc = load_dlc_txt(os.path.join(DIRETORIO_DLC_TXT, info_dlc))
        if df_dlc is None:
            continue
        df_dlc = add_time_column(df_dlc, 'Frame', FPS_VIDEO)
        df_dlc['exploration_type'] = df_dlc.apply(
            determine_dlc_exploration, axis=1, definicoes=rois, col_x='X', col_y='Y')

        periods_ts  = _generate_periods(df_ts)
        periods_dlc = _generate_periods(df_dlc)

        # Calculate times
        times = {s: {'objeto_Familiar': 0, 'objeto_Novo': 0}
                 for s in ['topscan', 'dlc']}
        for system, periods in [('topscan', periods_ts), ('dlc', periods_dlc)]:
            for _, dur, kind in periods:
                if kind in times[system]:
                    times[system][kind] += dur

        saved_times[rat_id] = {
            "topscan_novo": times['topscan']['objeto_Novo'],
            "topscan_familiar": times['topscan']['objeto_Familiar'],
            "dlc_novo": times['dlc']['objeto_Novo'],
            "dlc_familiar": times['dlc']['objeto_Familiar'],
        }

        # --- Broken Bar ---
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 5), sharex=True, constrained_layout=True)
        for ax, periods in [(ax1, periods_ts), (ax2, periods_dlc)]:
            for start, dur, kind in periods:
                ax.broken_barh([(start, dur)], (0, 1), facecolors=colors.get(kind, '#808080'))

        max_time = max(df_ts['time'].max() if not df_ts.empty else 0,
                       df_dlc['time'].max() if not df_dlc.empty else 0)
        for ax, name in [(ax1, 'TopScan'), (ax2, 'DeepLabCut')]:
            ax.set_ylim(0, 1); ax.set_yticks([0.5]); ax.set_yticklabels([name], fontsize=15)
            ax.grid(True, axis='x', linestyle='--', alpha=0.7)
            ax.set_xlim(0, max_time if max_time > 0 else 300)

        ax1.set_title(f' ')
        ax2.set_xlabel('Time (seconds)', fontsize=15)
        patches = [mpatches.Patch(color=colors[k], label=k.replace('objeto_', 'Object ').capitalize())
                   for k in colors]
        fig.legend(handles=patches, loc='upper right', ncol=len(patches))
        plt.suptitle(f"Behavioural Exploration — {rat_id}", fontsize=16)
        plt.show()

    return saved_times

##STAGE 4: STATISTICS

###STAGE 4.1: Trajectory similarity metrics (cm and pixels)

In [ ]:
def run_statistics(common_rats, results_cm, dlc_files, defined_rois, saved_times):
    print("\n=== 4.1 Similarity Metrics ===")
    metrics_results = []

    for rat_id in sorted(common_rats):
        df_analysis = _sync_data_cm(rat_id, results_cm)
        if df_analysis is None:
            continue

        traj_ts  = df_analysis[['x_cm_topscan', 'y_cm_topscan']].values
        traj_dlc = df_analysis[['x_cm_dlc', 'y_cm_dlc']].values

        df_analysis['distance_cm'] = np.sqrt(
            (df_analysis['x_cm_dlc'] - df_analysis['x_cm_topscan'])**2 +
            (df_analysis['y_cm_dlc'] - df_analysis['y_cm_topscan'])**2
        )

        rmse    = np.sqrt(np.mean(df_analysis['distance_cm']**2))
        mae     = df_analysis['distance_cm'].mean()
        std_err = df_analysis['distance_cm'].std()
        frechet = similaritymeasures.frechet_dist(traj_dlc, traj_ts)
        dtw, _  = similaritymeasures.dtw(traj_dlc, traj_ts)
        corr_x, _ = pearsonr(df_analysis['x_cm_dlc'], df_analysis['x_cm_topscan'])
        corr_y, _ = pearsonr(df_analysis['y_cm_dlc'], df_analysis['y_cm_topscan'])

        metrics_results.append({
            "Animal ID": rat_id, "RMSE (cm)": rmse, "MAE (cm)": mae,
            "Std Error (cm)": std_err, "Fréchet (cm)": frechet,
            "DTW (cm)": dtw, "Corr X": corr_x, "Corr Y": corr_y,
        })

    if metrics_results:
        df_met = pd.DataFrame(metrics_results)
        pd.set_option('display.float_format', '{:.3f}'.format)
        print(df_met.to_string(index=False))

###STAGE 4.2: Error heatmap (cm)

In [ ]:
print("\n=== 4.2 Error Heatmaps ===")
    NUM_BINS = 50
    for rat_id in common_rats:
        df_analysis = _sync_data_cm(rat_id, results_cm)
        if df_analysis is None:
            continue
        df_analysis['distance_cm'] = np.sqrt(
            (df_analysis['x_cm_dlc'] - df_analysis['x_cm_topscan'])**2 +
            (df_analysis['y_cm_dlc'] - df_analysis['y_cm_topscan'])**2
        )
        bins_x = np.linspace(0, ARENA_LARGURA_CM, NUM_BINS + 1)
        bins_y = np.linspace(0, ARENA_ALTURA_CM,  NUM_BINS + 1)
        hm_sum, _, _ = np.histogram2d(df_analysis['x_cm_dlc'], df_analysis['y_cm_dlc'],
                                      bins=[bins_x, bins_y], weights=df_analysis['distance_cm'])
        hm_cnt, _, _ = np.histogram2d(df_analysis['x_cm_dlc'], df_analysis['y_cm_dlc'], bins=[bins_x, bins_y])
        hm_avg = np.divide(hm_sum, hm_cnt, out=np.zeros_like(hm_sum), where=hm_cnt != 0)

        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(hm_avg.T, origin='lower', cmap='jet',
                       extent=[0, ARENA_LARGURA_CM, 0, ARENA_ALTURA_CM])
        ax.plot(df_analysis['x_cm_dlc'], df_analysis['y_cm_dlc'], color='white', alpha=0.3, linewidth=1)
        cbar = fig.colorbar(im, ax=ax); cbar.set_label('Mean Error (cm)')
        ax.set_title(f'Error Heatmap — {rat_id}')
        ax.set_xlabel('X (cm)'); ax.set_ylabel('Y (cm)')
        plt.show()

###STAGE 4.3: Error by behavioural state (Repeated-measures ANOVA)

In [ ]:
print("\n=== 4.3 Error by Behavioural State ===")
    anova_data = []
    for rat_id in common_rats:
        df_analysis = _sync_data_cm(rat_id, results_cm)
        if df_analysis is None:
            continue
        df_analysis['distance_cm'] = np.sqrt(
            (df_analysis['x_cm_dlc'] - df_analysis['x_cm_topscan'])**2 +
            (df_analysis['y_cm_dlc'] - df_analysis['y_cm_topscan'])**2
        )
        info_dlc = dlc_files.get(BODYPART_LEGENDA, {}).get(rat_id)
        rois     = defined_rois.get(rat_id)
        if not info_dlc or not rois:
            continue
        df_comp = _prepare_dlc_data_for_video(
            os.path.join(DIRETORIO_DLC_TXT, info_dlc), FPS_VIDEO, object_definitions=rois)
        if df_comp is None:
            continue
        df_final = pd.merge(df_analysis, df_comp[['FrameNum', 'exploration_type']], on='FrameNum')
        if df_final.empty:
            continue

        # Boxplot per animal
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df_final, x='exploration_type', y='distance_cm',
                    palette=['#E0E0E0', '#9E9E9E', '#616161'])
        sns.swarmplot(data=df_final, x='exploration_type', y='distance_cm', color='black', alpha=0.6, size=4)
        plt.title(f'Error by Behavioural State — {rat_id}')
        plt.xlabel('State (DLC)'); plt.ylabel('Error (cm)')
        plt.tight_layout(); plt.show()

        for cond, err in df_final.groupby('exploration_type')['distance_cm'].mean().items():
            anova_data.append({'sujeito': rat_id, 'condicao': cond, 'erro_medio_cm': err})

    if anova_data:
        df_av = pd.DataFrame(anova_data)
        print("\n--- Repeated-Measures ANOVA ---")
        try:
            aov = pg.rm_anova(data=df_av, dv='erro_medio_cm', within='condicao', subject='sujeito', detailed=True)
            print(aov.round(4))
            p_val = aov.loc[0, 'p-GG-corr'] if aov.loc[0, 'p-spher'] < 0.05 else aov.loc[0, 'p-unc']
            if p_val < 0.05:
                posthoc = pg.pairwise_tests(data=df_av, dv='erro_medio_cm',
                                             within='condicao', subject='sujeito', padjust='bonf')
                print("\nPost-hoc (Bonferroni):")
                print(posthoc.round(4))
        except Exception as e:
            print(f"ANOVA failed: {e}")

        # Aggregated bar plot
        label_map = {'chao': 'Floor', 'objeto_Familiar': 'Familiar Object', 'objeto_Novo': 'Novel Object'}
        df_av['condition_en'] = df_av['condicao'].map(label_map)
        ordem = [v for v in ['Floor', 'Familiar Object', 'Novel Object'] if v in df_av['condition_en'].values]
        plt.figure(figsize=(10, 7))
        sns.barplot(data=df_av, x='condition_en', y='erro_medio_cm', order=ordem,
                    palette=['#E0E0E0', '#9E9E9E', '#616161'], errorbar='se',
                    capsize=0.1, edgecolor='.1', linewidth=1.5)
        sns.swarmplot(data=df_av, x='condition_en', y='erro_medio_cm',
                      order=ordem, color='black', alpha=0.7, size=6)
        plt.xlabel('Behavioral State', fontweight='bold')
        plt.ylabel('Mean Distance Error (cm)', fontweight='bold')
        plt.tight_layout(); plt.show()

###STAGE 4.4: Discrimination Index + Shapiro + t-test

In [ ]:
print("\n=== 4.4 Discrimination Index ===")
    di_ts_list, di_dlc_list, ratos_di = [], [], []
    for rat_id, times in saved_times.items():
        tt_ts  = times["topscan_novo"] + times["topscan_familiar"]
        tt_dlc = times["dlc_novo"]     + times["dlc_familiar"]
        if tt_ts > 0 and tt_dlc > 0:
            di_ts_list.append((times["topscan_novo"] - times["topscan_familiar"]) / tt_ts)
            di_dlc_list.append((times["dlc_novo"]    - times["dlc_familiar"])     / tt_dlc)
            ratos_di.append(rat_id)
            print(f"  {rat_id} | DI TopScan={di_ts_list[-1]:.4f}  DI DLC={di_dlc_list[-1]:.4f}")

    if len(ratos_di) >= 3:
        differences = np.array(di_dlc_list) - np.array(di_ts_list)
        stat_sw, p_sw = shapiro(differences)
        print(f"\nShapiro-Wilk (DI differences): W={stat_sw:.4f}, p={p_sw:.4f}")
        t_stat, p_t = ttest_rel(di_ts_list, di_dlc_list)
        print(f"Paired t-test: t={t_stat:.4f}, p={p_t:.4f}")

        fig, ax = plt.subplots(figsize=(8, 8), layout='constrained')
        ax.scatter(di_ts_list, di_dlc_list, c='black', s=50, alpha=0.7)
        lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
        ax.plot(lims, lims, 'r--', alpha=0.75, label='Ideal Agreement (y=x)')
        ax.set_xlabel('DI TopScan', fontsize=15); ax.set_ylabel('DI DeepLabCut', fontsize=15)
        ax.legend(); ax.set_aspect('equal', adjustable='box')
        ax.grid(True, linestyle='--')
        plt.show()

###STAGE 4.5: Bland-Altman (Total exploration time)

In [ ]:
print("\n=== 4.5 Bland-Altman ===")
    t_ts_arr  = np.array([v["topscan_novo"] + v["topscan_familiar"] for v in saved_times.values()])
    t_dlc_arr = np.array([v["dlc_novo"]     + v["dlc_familiar"]     for v in saved_times.values()])
    valid     = (t_ts_arr > 0) & (t_dlc_arr > 0)
    t_ts_arr, t_dlc_arr = t_ts_arr[valid], t_dlc_arr[valid]

    if len(t_ts_arr) >= 2:
        mean_val = (t_ts_arr + t_dlc_arr) / 2
        diff     = t_ts_arr - t_dlc_arr
        bias     = np.mean(diff); sd = np.std(diff, ddof=1)
        print(f"Bias: {bias:.2f} s | Limits ±1.96 SD: [{bias - 1.96*sd:.2f}, {bias + 1.96*sd:.2f}] s")

        fig, ax = plt.subplots(figsize=(10, 7), layout='constrained')
        ax.scatter(mean_val, diff, c='black', s=50, alpha=0.7)
        ax.axhline(bias,             color='red',  linestyle='--', label=f'Bias: {bias:.2f}')
        ax.axhline(bias + 1.96 * sd, color='gray', linestyle=':',  label='±1.96 SD')
        ax.axhline(bias - 1.96 * sd, color='gray', linestyle=':')
        ax.axhline(0, color='black', linewidth=0.5)
        ax.set_xlabel('Mean Exploration Time (s)')
        ax.set_ylabel('Difference (TopScan − DLC) (s)')
        ax.legend(); ax.grid(True, linestyle='--', alpha=0.6)
        plt.show()

###STAGE 4.6: Temporal cross-correlation (Group)

In [ ]:
print("\n=== 4.6 Temporal Cross-Correlation ===")
    WINDOW = 100
    group_lags, curves = [], []

    for rat_id in common_rats:
        df_analysis = _sync_data_cm(rat_id, results_cm)
        if df_analysis is None or len(df_analysis) < 20:
            continue
        sig_dlc = (df_analysis['x_cm_dlc']     - df_analysis['x_cm_dlc'].mean()).values
        sig_ts  = (df_analysis['x_cm_topscan'] - df_analysis['x_cm_topscan'].mean()).values
        corr    = correlate(sig_dlc * hann(len(sig_dlc)), sig_ts * hann(len(sig_ts)), mode='full')
        lags    = np.arange(-len(sig_ts) + 1, len(sig_ts))
        lag_ot  = int(lags[np.argmax(corr)])
        group_lags.append({'Rat_ID': rat_id, 'Optimal_Lag': lag_ot})

        idx0 = np.where(lags == 0)[0][0]
        ini, fim = idx0 - WINDOW, idx0 + WINDOW
        if ini >= 0 and fim < len(corr):
            segment = corr[ini:fim]
            curves.append(segment / np.max(np.abs(segment)))

    if group_lags:
        df_lags   = pd.DataFrame(group_lags)
        mean_lag  = df_lags['Optimal_Lag'].mean()
        std_lag   = df_lags['Optimal_Lag'].std()
        print(f"Group mean lag: {mean_lag:.2f} ± {std_lag:.2f} frames")

        if curves:
            matrix     = np.array(curves)
            mean_curve = matrix.mean(axis=0)
            sem_curve  = matrix.std(axis=0) / np.sqrt(len(curves))
            x_axis     = np.arange(-WINDOW, WINDOW)

            plt.figure(figsize=(12, 7))
            plt.plot(x_axis, mean_curve, color='#1f77b4', linewidth=2, label='Mean Cross-Correlation')
            plt.fill_between(x_axis, mean_curve - sem_curve, mean_curve + sem_curve,
                             color='#1f77b4', alpha=0.2, label='SEM')
            plt.axvline(0,        color='red',   linestyle='--', linewidth=2, label='Zero Lag')
            plt.axvline(mean_lag, color='green', linestyle=':',  linewidth=2,
                        label=f'Group Lag ({mean_lag:.1f} fr)')
            plt.xlabel('Lag (Frames)'); plt.ylabel('Normalised Correlation')
            plt.legend(); plt.grid(True, linestyle='--', alpha=0.7)
            plt.tight_layout(); plt.show()

        plt.figure(figsize=(7, 6))
        sns.boxplot(y=df_lags['Optimal_Lag'], color='lightgray', width=0.4, showfliers=False)
        sns.swarmplot(y=df_lags['Optimal_Lag'], color='black', size=8, alpha=0.7)
        plt.axhline(0, color='red', linestyle='--', linewidth=2, label='Perfect Synchronisation')
        plt.ylabel('Optimal Lag (Frames)'); plt.xticks([])
        plt.tight_layout(); plt.show()

###STAGE 4.7: Spatial analysis: Centre vs. Periphery

In [ ]:
print("\n=== 4.7 Error Centre vs. Periphery ===")
    MARGIN = 0.20
    for rat_id in common_rats:
        df_analysis = _sync_data_cm(rat_id, results_cm)
        if df_analysis is None:
            continue
        df_analysis['distance_cm'] = np.sqrt(
            (df_analysis['x_cm_dlc'] - df_analysis['x_cm_topscan'])**2 +
            (df_analysis['y_cm_dlc'] - df_analysis['y_cm_topscan'])**2
        )
        centre = (
            df_analysis['x_cm_dlc'].between(ARENA_LARGURA_CM * MARGIN, ARENA_LARGURA_CM * (1 - MARGIN)) &
            df_analysis['y_cm_dlc'].between(ARENA_ALTURA_CM  * MARGIN, ARENA_ALTURA_CM  * (1 - MARGIN))
        )
        df_analysis['zone'] = np.where(centre, 'Centre', 'Periphery')
        plt.figure(figsize=(8, 6))
        sns.boxplot(data=df_analysis, x='zone', y='distance_cm', palette='plasma')
        plt.title(f'Error by Zone — {rat_id}')
        plt.xlabel('Zone'); plt.ylabel('Error (cm)')
        plt.grid(True, linestyle='--', alpha=0.6, axis='y')
        plt.tight_layout(); plt.show()

###STAGE 4.8: Velocity profiles (KDE + K-S)

In [ ]:
print("\n=== 4.8 Velocity Profiles ===")
    for rat_id in common_rats:
        df_analysis = _sync_data_cm(rat_id, results_cm)
        if df_analysis is None:
            continue
        for sys, cx, cy, ct in [('dlc', 'x_cm_dlc', 'y_cm_dlc', 'time_dlc'),
                                  ('topscan', 'x_cm_topscan', 'y_cm_topscan', 'time_topscan')]:
            df_analysis[f'vel_{sys}'] = (np.sqrt(df_analysis[cx].diff()**2 + df_analysis[cy].diff()**2) /
                                         df_analysis[ct].diff())
        df_analysis = df_analysis.replace([np.inf, -np.inf], np.nan).dropna(subset=['vel_dlc', 'vel_topscan'])
        if df_analysis.empty:
            continue
        ks, pv = ks_2samp(df_analysis['vel_dlc'], df_analysis['vel_topscan'])
        print(f"  {rat_id} | K-S D={ks:.3f}, p={pv:.3g}")

        lim_x = df_analysis['vel_dlc'].quantile(0.995)
        plt.figure(figsize=(10, 6))
        sns.kdeplot(df_analysis['vel_dlc'],     label='DLC',     color='purple', fill=True, alpha=0.3)
        sns.kdeplot(df_analysis['vel_topscan'], label='TopScan', color='green',  fill=True, alpha=0.3)
        plt.xlabel('Velocity (cm/s)'); plt.ylabel('Density')
        plt.xlim(0, lim_x); plt.legend(); plt.grid(True, linestyle='--')
        plt.title(f'Velocity Distribution — {rat_id}')
        plt.tight_layout(); plt.show()

###STAGE 4.9: Dashboard: trajectory + error × time + heatmap + error × velocity

In [ ]:
print("\n=== 4.9 Validation Dashboard ===")
    NUM_BINS_DASH = 50
    for rat_id in common_rats:
        df_analysis = _sync_data_cm(rat_id, results_cm)
        if df_analysis is None:
            continue
        df_analysis['distance_cm'] = np.sqrt(
            (df_analysis['x_cm_dlc'] - df_analysis['x_cm_topscan'])**2 +
            (df_analysis['y_cm_dlc'] - df_analysis['y_cm_topscan'])**2
        )
        df_analysis['vel_cm_s'] = (np.sqrt(df_analysis['x_cm_dlc'].diff()**2 + df_analysis['y_cm_dlc'].diff()**2) /
                                   df_analysis['time_dlc'].diff())
        df_analysis = df_analysis.replace([np.inf, -np.inf], np.nan).dropna(subset=['vel_cm_s', 'distance_cm'])

        rmse          = np.sqrt(np.mean(df_analysis['distance_cm']**2))
        corr_vel_e, _ = pearsonr(df_analysis['vel_cm_s'], df_analysis['distance_cm'])

        fig, axes = plt.subplots(2, 2, figsize=(18, 16), layout='constrained')
        fig.suptitle(f'Validation Dashboard — {rat_id}\n'
                     f'RMSE={rmse:.2f} cm  |  Corr(Error, Vel)={corr_vel_e:.2f}', fontsize=22)

        # Panel 1: trajectories
        ax = axes[0, 0]
        ax.plot(df_analysis['x_cm_dlc'], df_analysis['y_cm_dlc'],         color='purple', alpha=0.6, label='DLC')
        ax.plot(df_analysis['x_cm_topscan'], df_analysis['y_cm_topscan'], color='green',  alpha=0.6, linestyle='--', label='TopScan')
        ax.set_xlim(0, ARENA_LARGURA_CM); ax.set_ylim(0, ARENA_ALTURA_CM)
        ax.set_aspect('equal', adjustable='box')
        ax.set_title('Trajectory Alignment', fontsize=19, fontweight='bold')
        ax.set_xlabel('X (cm)', fontsize=15); ax.set_ylabel('Y (cm)', fontsize=15)
        ax.legend(fontsize=13); ax.grid(True, linestyle=':')

        # Panel 2: error × time
        ax = axes[0, 1]
        ax.plot(df_analysis['FrameNum'], df_analysis['distance_cm'], color='blue', alpha=0.7)
        mean_e = df_analysis['distance_cm'].mean()
        ax.axhline(mean_e, color='orange', linestyle='--', label=f'Mean ({mean_e:.2f} cm)')
        ax.set_title('Positional Error × Time', fontsize=19, fontweight='bold')
        ax.set_xlabel('Frame', fontsize=15); ax.set_ylabel('Error (cm)', fontsize=15)
        ax.legend(fontsize=13); ax.grid(True, linestyle=':')

        # Panel 3: heatmap
        ax = axes[1, 0]
        bins_x = np.linspace(0, ARENA_LARGURA_CM, NUM_BINS_DASH + 1)
        bins_y = np.linspace(0, ARENA_ALTURA_CM,  NUM_BINS_DASH + 1)
        hm_s, _, _ = np.histogram2d(df_analysis['x_cm_dlc'], df_analysis['y_cm_dlc'],
                                     bins=[bins_x, bins_y], weights=df_analysis['distance_cm'])
        hm_c, _, _ = np.histogram2d(df_analysis['x_cm_dlc'], df_analysis['y_cm_dlc'], bins=[bins_x, bins_y])
        hm_avg = np.divide(hm_s, hm_c, out=np.zeros_like(hm_s), where=hm_c != 0)
        im = ax.imshow(hm_avg.T, origin='lower', cmap='jet',
                       extent=[0, ARENA_LARGURA_CM, 0, ARENA_ALTURA_CM])
        cbar = fig.colorbar(im, ax=ax); cbar.set_label('Mean Error (cm)', fontsize=12)
        ax.set_title('Error Heatmap', fontsize=19, fontweight='bold')
        ax.set_xlabel('X (cm)', fontsize=15); ax.set_ylabel('Y (cm)', fontsize=15)

        # Panel 4: error × velocity
        ax = axes[1, 1]
        ax.scatter(df_analysis['vel_cm_s'], df_analysis['distance_cm'], alpha=0.2, s=10)
        m, b = np.polyfit(df_analysis['vel_cm_s'], df_analysis['distance_cm'], 1)
        x_l = np.linspace(df_analysis['vel_cm_s'].min(), df_analysis['vel_cm_s'].max(), 100)
        ax.plot(x_l, m * x_l + b, color='red', linewidth=2)
        r_v, _ = pearsonr(df_analysis['vel_cm_s'], df_analysis['distance_cm'])
        ax.text(0.05, 0.9, f'r = {r_v:.2f}', transform=ax.transAxes,
                fontweight='bold', color='red', fontsize=12)
        ax.set_title('Error × Velocity', fontsize=19, fontweight='bold')
        ax.set_xlabel('Velocity (cm/s)', fontsize=15); ax.set_ylabel('Error (cm)', fontsize=15)
        ax.grid(True, linestyle=':')

        # layout='constrained' handles spacing automatically
        plt.show()

#ENTRY POINT — runs the complete pipeline

In [ ]:
def run_pipeline():
    # Stage 0
    common_rats, topscan_files, dlc_files = load_data()

    # Stage 1
    results_cm = process_trajectories(common_rats, topscan_files, dlc_files)

    # Stages 1.2 and 1.3
    plot_comparative_trajectories(results_cm)
    plot_raw_trajectories(common_rats, topscan_files, dlc_files)

    # Stage 2
    defined_rois = define_rois(common_rats)
    verify_rois(defined_rois, topscan_files)
    generate_videos(common_rats, topscan_files, dlc_files, results_cm, defined_rois)

    # Stage 3
    saved_times = calculate_behavioral_exploration(
        common_rats, topscan_files, dlc_files, defined_rois)

    # Stage 4
    run_statistics(common_rats, results_cm, dlc_files, defined_rois, saved_times)


if __name__ == '__main__':
    run_pipeline()